# Recurrence Basis — 2D Visualization

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# ---------------------------------------------------------------------------
# 1) Shared learnable recurrence:
#       R_0 = 0
#       R_1 = 1
#       R_{n+1} = (a h^2 + b h + c) R_n + (d h + e) R_{n-1}
#    a, b, c, d, e are ONE set of parameters shared by the whole network
#    (init close to Chebyshev-U: a=0, b=2, c=0, d=-1, e=0)
# ---------------------------------------------------------------------------
class RecurrenceBasis(nn.Module):
    def __init__(self, degree, init=(0.0, 2.0, 0.0, -1.0, 0.0)):
        super().__init__()
        self.degree = degree
        a, b, c, d, e = init
        self.a = nn.Parameter(torch.tensor(float(a)))
        self.b = nn.Parameter(torch.tensor(float(b)))
        self.c = nn.Parameter(torch.tensor(float(c)))
        self.d = nn.Parameter(torch.tensor(float(d)))
        self.e = nn.Parameter(torch.tensor(float(e)))

    def forward(self, h):
        # h: (..., ) elementwise, already normalized to roughly [-1, 1]
        coef2 = self.a * h ** 2 + self.b * h + self.c
        coef1 = self.d * h + self.e

        R_prev2 = torch.zeros_like(h)          # R_0
        R_prev1 = torch.ones_like(h)           # R_1
        basis = [R_prev2, R_prev1]
        for _ in range(self.degree - 1):
            R_next = coef2 * R_prev1 + coef1 * R_prev2
            basis.append(R_next)
            R_prev2, R_prev1 = R_prev1, R_next
        return torch.stack(basis, dim=-1)      # (..., degree+1)


class RecKANLayer(nn.Module):
    def __init__(self, input_dim, output_dim, degree, basis: RecurrenceBasis):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.degree = degree
        self.basis = basis  # shared instance -> a..e shared across layers
        self.coeffs = nn.Parameter(
            torch.randn(input_dim, output_dim, degree + 1)
            / (input_dim * (degree + 1)) ** 0.5
        )

    def forward(self, x):
        x = torch.tanh(x)                      # keep h in (-1, 1) for stability
        R = self.basis(x)                       # (batch, input_dim, degree+1)
        y = torch.einsum('bid,iod->bo', R, self.coeffs)
        return y


class RecKAN(nn.Module):
    def __init__(self, degree=8):
        super().__init__()
        self.basis = RecurrenceBasis(degree)    # <-- shared a,b,c,d,e
        self.layer1 = RecKANLayer(1, 8, degree, self.basis)
        self.layer2 = RecKANLayer(8, 1, degree, self.basis)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        return x


# ---------------------------------------------------------------------------
# 2) Baseline MLP
# ---------------------------------------------------------------------------
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(1, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.layers(x - 1)


# ---------------------------------------------------------------------------
# 3) Same noisy piecewise target function as before
# ---------------------------------------------------------------------------
def target_function(x):
    y = np.zeros_like(x)
    mask1 = x < 0.5
    y[mask1] = np.sin(20 * np.pi * x[mask1]) + x[mask1] ** 2
    mask2 = (0.5 <= x) & (x < 1.5)
    y[mask2] = 0.5 * x[mask2] * np.exp(-x[mask2]) + np.abs(np.sin(5 * np.pi * x[mask2]))
    mask3 = x >= 1.5
    y[mask3] = np.log(x[mask3] - 1) / np.log(2) - np.cos(2 * np.pi * x[mask3])
    noise = np.random.normal(0, 0.2, y.shape)
    y += noise
    return y


x_train = torch.linspace(0, 2, steps=500).unsqueeze(1)
y_train = torch.tensor(target_function(x_train.numpy()), dtype=torch.float32)

reckan_model = RecKAN(degree=8)
mlp_model = SimpleMLP()

criterion = nn.MSELoss()
opt_reckan = torch.optim.Adam(reckan_model.parameters(), lr=0.01)
opt_mlp = torch.optim.Adam(mlp_model.parameters(), lr=0.03)

epochs = 50000  # reduced from 200000 for a quick run; increase for full training
reckan_losses, mlp_losses = [], []

for epoch in range(epochs):
    opt_reckan.zero_grad()
    out_reckan = reckan_model(x_train)
    loss_reckan = criterion(out_reckan, y_train)
    loss_reckan.backward()
    opt_reckan.step()

    opt_mlp.zero_grad()
    out_mlp = mlp_model(x_train)
    loss_mlp = criterion(out_mlp, y_train)
    loss_mlp.backward()
    opt_mlp.step()

    if epoch % 1000 == 0:
        reckan_losses.append(loss_reckan.item())
        mlp_losses.append(loss_mlp.item())
        print(f'Epoch {epoch}/{epochs}  RecKAN loss: {loss_reckan.item():.4f}  '
              f'MLP loss: {loss_mlp.item():.4f}')

print('\nLearned recurrence parameters (a,b,c,d,e):')
b = reckan_model.basis
print(f'a={b.a.item():.4f}  b={b.b.item():.4f}  c={b.c.item():.4f}  '
      f'd={b.d.item():.4f}  e={b.e.item():.4f}')

# ---------------------------------------------------------------------------
# 4) Evaluate + plot
# ---------------------------------------------------------------------------
x_test = torch.linspace(0, 2, steps=400).unsqueeze(1)
y_true_clean = target_function(x_test.numpy().copy())  # noisy again just for plotting scatter
with torch.no_grad():
    y_pred_reckan = reckan_model(x_test)
    y_pred_mlp = mlp_model(x_test)

plt.figure(figsize=(10, 6))
plt.scatter(x_train.numpy(), y_train.numpy(), s=8, alpha=0.3, label='Noisy training data')
plt.plot(x_test.numpy(), y_pred_reckan.numpy(), 'r-', linewidth=2, label='RecKAN (learnable recurrence)')
plt.plot(x_test.numpy(), y_pred_mlp.numpy(), 'g-', linewidth=2, label='MLP')
plt.legend()
plt.title('Fitting a noisy piecewise function: RecKAN vs MLP')
plt.xlabel('x')
plt.ylabel('y')
plt.tight_layout()
plt.savefig('/home/claude/fit_comparison.png', dpi=130)

plt.figure(figsize=(8, 5))
steps_axis = np.arange(0, epochs, 1000)
plt.plot(steps_axis, reckan_losses, label='RecKAN loss')
plt.plot(steps_axis, mlp_losses, label='MLP loss')
plt.yscale('log')
plt.xlabel('epoch')
plt.ylabel('MSE loss (log scale)')
plt.legend()
plt.title('Training loss')
plt.tight_layout()
plt.savefig('/home/claude/loss_curve.png', dpi=130)

print('\nFinal RecKAN loss:', reckan_losses[-1])
print('Final MLP loss:', mlp_losses[-1])